# Auditable record reconciliation

Two systems export overlapping records, but one source lacks a reliable email key. A naive join drops matches, duplicates records or attaches one person's attributes to another.

Normalize identity fields, block candidates by email/name/IP, apply explicit evidence rules, require a separation margin and preserve a review queue. Conflicting emails, uncertain dates and multiple records claiming the same target never silently overwrite the source.

Rebuild source artifacts with `python scripts/run_all.py` from the repository root. This walkthrough inspects the measured run.

In [1]:
from pathlib import Path
import json
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
filename = "audit.json"
report = json.loads((root / "artifacts/reconciliation" / filename).read_text(encoding="utf-8"))
print(json.dumps(report.get("test", report.get("decisions", report.get("aggregate"))), indent=2))

{
  "review": 5,
  "matched": 2,
  "unmatched": 1
}


In [2]:
import pandas as pd
result = pd.read_csv(root / "artifacts/reconciliation/decisions.csv")
print(result.head(8).to_string(index=False))

right_id left_id    status  score  margin                              reason
    B001    A001   matched   1.00     1.0                rule_and_margin_pass
    B002    A002   matched   1.00     1.0                rule_and_margin_pass
    B003    A003    review   1.00     0.0  ambiguous_or_insufficient_evidence
    B004     NaN unmatched   0.00     1.0                       no_candidates
    B005    A005    review   1.00     1.0  ambiguous_or_insufficient_evidence
    B006    A007    review   0.85     1.0  ambiguous_or_insufficient_evidence
    B007    A008    review   1.00     1.0 multiple_right_records_for_one_left
    B008    A008    review   1.00     1.0 multiple_right_records_for_one_left


## Interpretation and limits

Synthetic fixtures exercise edge cases. Evidence scores are policy weights, not calibrated identity probabilities. Matching quality on a real dataset requires reviewed ground truth and an agreed matching policy.